In [1]:
import numpy as np
import pandas as pd
import joblib
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report
)

BASE = r'C:\Users\Anushka Thite\Desktop\smart-loan-system'
PROCESSED_DIR = os.path.join(BASE, 'data', 'processed')
MODELS_DIR = os.path.join(BASE, 'models')

X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
y_test = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

base_models = [
    ('xgb', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          random_state=42, eval_metric='logloss', n_jobs=-1)),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

stack = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

print("Training Stacking (5-fold CV)...")
start = time.time()
stack.fit(X_train, y_train)
train_time = time.time() - start

y_pred = stack.predict(X_test)
y_proba = stack.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"\n{'=' * 55}")
print(f"STACKING RESULTS")
print(f"{'=' * 55}")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Time     : {train_time:.2f} sec")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

joblib.dump(stack, os.path.join(MODELS_DIR, 'stacking.pkl'))
print(f"\n✅ Saved to models/stacking.pkl")

Train: (20570, 63), Test: (4000, 63)

Training Stacking (5-fold CV)...

STACKING RESULTS
Accuracy : 0.9523
Precision: 0.9556
Recall   : 0.9708
F1       : 0.9631
AUC      : 0.9914
Time     : 31.69 sec

Classification Report:
              precision    recall  f1-score   support

    Rejected       0.95      0.92      0.93      1429
    Approved       0.96      0.97      0.96      2571

    accuracy                           0.95      4000
   macro avg       0.95      0.94      0.95      4000
weighted avg       0.95      0.95      0.95      4000


✅ Saved to models/stacking.pkl
